# HairSeg_Train — 헤어라인 분할 모델 직접 학습 (CelebAMask-HQ, 4클래스)

구글 `selfie_multiclass_256x256.tflite` 대신 쓸 **우리 분할 모델**을 만듭니다. 픽셀마다 0 배경 / 1 피부 / 2 머리카락 / 3 모자 를 붙이는 작은 U-Net.
절차 설명: `FaceReading_3/CelebAMask_헤어라인_학습_안내.html`

**만드는 것 (보드로 복사)**: `save/hair_seg.tflite` (+ 발표용 `hairseg_samples.png`, `hairseg_train_log.png`)

**미리 준비할 것**
- Kaggle **CelebAMask-HQ** (`ipythonx/celebamaskhq`) zip 전체 → 드라이브 `ai_semiconductor/Python/examples/files/CelebAMask-HQ.zip` (이름이 archive.zip 이어도 찾음)
- 런타임: **GPU(T4)**. 압축 해제 5분 + 전처리 10분 + 학습 40분.
- (선택, 8절 교차검증용) 드라이브 `proj_face/FaceReading_3/` 에 `wflw_prep.py`, `wflw_mesh.py`, `save/` 에 `wflw_mesh.tflite`, `face_detection_yunet_2023mar.onnx`


## 1. 드라이브 · 경로

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive'); colab = True
except Exception:
    colab = False

In [ ]:
import os, sys, glob, time, json
if colab:
    BASE = '/content/drive/MyDrive/ai_semiconductor/Python/examples/'
    CODE_DIR = BASE + 'proj_face/FaceReading_3/'; save_dir = BASE + 'files/save/'; work_dir = '/content/hairseg'
    cands = glob.glob(BASE + 'files/CelebAMask-HQ*.zip') + glob.glob(BASE + 'files/archive.zip')
else:
    CODE_DIR = './'; save_dir = '../files/save/'; work_dir = './hairseg'; cands = glob.glob('../files/CelebAMask-HQ*.zip')
assert cands, 'files/ 에 CelebAMask-HQ zip 이 없습니다'
dataset_zip = cands[0]; print('zip:', dataset_zip, '%.2f GB' % (os.path.getsize(dataset_zip) / 1e9))
os.makedirs(work_dir, exist_ok=True); os.makedirs(save_dir, exist_ok=True)

## 2. 압축 해제 · 폴더 찾기

In [ ]:
import subprocess
if not glob.glob(work_dir + '/**/CelebA-HQ-img', recursive=True):
    print('압축 해제 중 (3GB, 약 5분) …'); subprocess.run(['unzip', '-q', '-o', dataset_zip, '-d', work_dir], check=True)
IMG_DIR = glob.glob(work_dir + '/**/CelebA-HQ-img', recursive=True)[0] + '/'
ANNO    = glob.glob(work_dir + '/**/CelebAMask-HQ-mask-anno', recursive=True)[0] + '/'
print(IMG_DIR, len(os.listdir(IMG_DIR)), '장'); print(ANNO, sorted(os.listdir(ANNO))[:5], '…')
print('예시 마스크 파일:', sorted(os.listdir(ANNO + '0'))[:6])

## 3. 전처리: 부위별 PNG → 4클래스 맵 (256×256)
사진은 1024, 마스크는 512 라 둘 다 256 으로 맞춥니다. 마스크 축소는 반드시 `INTER_NEAREST`. 겹치는 픽셀은 모자 > 머리카락 > 피부 순으로 덮어씁니다.

In [ ]:
import numpy as np, cv2
S = 256                       # 모델 입력 크기 (보드 Segmenter 의 SEG_SIZE 와 같아야 함)
N = 10000                     # 사용할 사진 수 (RAM 12GB 기준 1만 장 ≈ 2.6GB). 더 쓰려면 늘리기
CLASSES = [('skin', 1), ('hair', 2), ('hat', 3)]      # 뒤가 우선 (덮어씀)
CLASS_NAMES = ['bg', 'skin', 'hair', 'hat']

def class_map(idx):
    m = np.zeros((512, 512), np.uint8); folder = ANNO + '%d/' % (idx // 2000)
    for name, cid in CLASSES:
        p = folder + '%05d_%s.png' % (idx, name)
        if os.path.exists(p): m[cv2.imread(p, cv2.IMREAD_GRAYSCALE) > 0] = cid
    return cv2.resize(m, (S, S), interpolation=cv2.INTER_NEAREST)

cache = work_dir + '/hairseg_%d_%d.npz' % (S, N)
if os.path.exists(cache):
    d = np.load(cache); X, Y = d['X'], d['Y']
else:
    X = np.zeros((N, S, S, 3), np.uint8); Y = np.zeros((N, S, S), np.uint8); t0 = time.time()
    for i in range(N):
        X[i] = cv2.resize(cv2.imread(IMG_DIR + '%d.jpg' % i), (S, S), interpolation=cv2.INTER_AREA)
        Y[i] = class_map(i)
        if i % 1000 == 0: print(i, '%.0fs' % (time.time() - t0))
    np.savez_compressed(cache, X=X, Y=Y)
print(X.shape, Y.shape)
print('클래스별 픽셀 비율:', {n: round(float((Y == c).mean()), 3) for c, n in enumerate(CLASS_NAMES)}, '(hat 1~3% 가 정상)')

In [ ]:
import matplotlib.pyplot as plt
PALETTE = np.array([[0, 0, 0], [80, 200, 255], [255, 120, 40], [200, 60, 200]], np.uint8)   # bg, skin, hair, hat (RGB)
def overlay(img_bgr, m, alpha=0.45):
    rgb = img_bgr[:, :, ::-1].astype(np.float32); col = PALETTE[m].astype(np.float32)
    out = rgb * (1 - alpha) + col * alpha; out[m == 0] = rgb[m == 0]
    return out.astype(np.uint8)
def show_rows(imgs, masks, preds=None, n=6, title=None, save=None):
    rows = 2 if preds is None else 3
    plt.figure(figsize=(2.2 * n, 2.3 * rows))
    for i in range(n):
        plt.subplot(rows, n, i + 1); plt.imshow(imgs[i][:, :, ::-1]); plt.axis('off')
        plt.subplot(rows, n, n + i + 1); plt.imshow(overlay(imgs[i], masks[i])); plt.axis('off')
        if preds is not None: plt.subplot(rows, n, 2 * n + i + 1); plt.imshow(overlay(imgs[i], preds[i])); plt.axis('off')
    if title: plt.suptitle(title)
    if save: plt.savefig(save, dpi=130, bbox_inches='tight')
    plt.show()
show_rows(X, Y, title='위: 사진 / 아래: 정답 마스크 (파랑 피부, 주황 머리카락, 보라 모자)')

## 4. 모델 (MobileNetV2 인코더 + U-Net 디코더) · 학습

In [ ]:
import tensorflow as tf
ALPHA = 0.35                                  # 라즈베리파이용. 0.5 로 올리면 경계가 조금 더 정확, 속도 1.5배 느림
base = tf.keras.applications.MobileNetV2((S, S, 3), include_top=False, weights='imagenet', alpha=ALPHA)
skip_names = ['block_1_expand_relu', 'block_3_expand_relu', 'block_6_expand_relu', 'block_13_expand_relu']   # 128, 64, 32, 16
skips = [base.get_layer(n).output for n in skip_names]
x = base.output                                                                  # 8×8
for s, f in zip(reversed(skips), (96, 64, 32, 16)):
    x = tf.keras.layers.UpSampling2D()(x)
    x = tf.keras.layers.Concatenate()([x, s])
    x = tf.keras.layers.SeparableConv2D(f, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.UpSampling2D()(x)                                            # 256×256
out = tf.keras.layers.Conv2D(4, 1, activation='softmax')(x)
model = tf.keras.Model(base.input, out)
model.compile(tf.keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.MeanIoU(num_classes=4, sparse_y_pred=False, name='miou')])
print('params: %.2fM' % (model.count_params() / 1e6))

In [ ]:
def gen(X, Y, bs=16):
    n = len(X)
    while True:
        idx = np.random.permutation(n)
        for i in range(0, n - bs + 1, bs):
            xb = X[idx[i:i + bs]].astype(np.float32); yb = Y[idx[i:i + bs]].copy()
            flip = np.random.rand(bs) < 0.5                                     # 좌우반전 (마스크도 같이)
            xb[flip] = xb[flip, :, ::-1]; yb[flip] = yb[flip, :, ::-1]
            xb = xb * np.random.uniform(0.7, 1.3, (bs, 1, 1, 1)) + np.random.uniform(-25, 25, (bs, 1, 1, 1))
            yield np.clip(xb, 0, 255)[:, :, :, ::-1] / 255., yb                  # BGR→RGB, 0~1  (보드 Segmenter 와 동일)

NVAL = 1000
Xtr, Ytr, Xva, Yva = X[:-NVAL], Y[:-NVAL], X[-NVAL:], Y[-NVAL:]
Xva_in = Xva[:, :, :, ::-1].astype(np.float32) / 255.

In [ ]:
%%time
EPOCHS = 20
hist = model.fit(gen(Xtr, Ytr), steps_per_epoch=len(Xtr) // 16, epochs=EPOCHS,
                 validation_data=(Xva_in, Yva),
                 callbacks=[tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, verbose=1),
                            tf.keras.callbacks.ModelCheckpoint(work_dir + '/hairseg_best.keras', save_best_only=True, monitor='val_miou', mode='max')])

## 5. 평가: 클래스별 IoU · 헤어라인 y 오차 · 그림

In [ ]:
model = tf.keras.models.load_model(work_dir + '/hairseg_best.keras')
P = model.predict(Xva_in, batch_size=32, verbose=0).argmax(-1).astype(np.uint8)
ious = {}
for c, name in enumerate(CLASS_NAMES):
    inter = ((P == c) & (Yva == c)).sum(); union = ((P == c) | (Yva == c)).sum()
    ious[name] = inter / max(union, 1); print('%-5s IoU %.3f' % (name, ious[name]))
print('mIoU %.3f' % np.mean(list(ious.values())))

def hairline_y(mask, skin=1):
    """가운데 열 20% 에서, 화면 중앙(피부)에서 위로 올라가며 피부가 끊기는 y 의 중앙값 (보드 hairline_from_mask 와 같은 생각)"""
    H, W = mask.shape; ys = []
    for x in range(int(W * 0.4), int(W * 0.6)):
        col = mask[:, x] == skin
        if not col[H // 2]: continue
        y = H // 2
        while y > 0 and col[y - 1]: y -= 1
        ys.append(y)
    return float(np.median(ys)) if ys else None
err = []
for i in range(len(Yva)):
    a, b = hairline_y(P[i]), hairline_y(Yva[i])
    if a is not None and b is not None: err.append(abs(a - b))
print('헤어라인 y 오차 (256px 기준): 평균 %.1f px, 중앙값 %.1f px, 측정 가능 %d/%d' % (np.mean(err), np.median(err), len(err), len(Yva)))

plt.figure(figsize=(6, 4)); plt.plot(hist.history['miou'], label='train'); plt.plot(hist.history['val_miou'], label='val')
plt.title('hair/skin segmentation  mIoU'); plt.xlabel('epoch'); plt.grid(True); plt.legend()
plt.savefig(save_dir + 'hairseg_train_log.png', dpi=150, bbox_inches='tight'); plt.show()
idx = np.random.choice(len(Xva), 6, replace=False)
show_rows(Xva[idx], Yva[idx], P[idx], title='위: 사진 / 가운데: 정답 / 아래: 우리 모델', save=save_dir + 'hairseg_samples.png')

### (선택) 구글 selfie_multiclass 와 같은 검증셋에서 비교
구글 모델은 클래스 번호가 다릅니다(1 머리카락, 3 얼굴 피부). 우리 번호로 바꿔서 IoU 를 나란히 봅니다.

In [ ]:
COMPARE_GOOGLE = True
if COMPARE_GOOGLE:
    import urllib.request
    gfile = 'selfie_multiclass_256x256.tflite'
    if not os.path.exists(gfile):
        src = save_dir + gfile if os.path.exists(save_dir + gfile) else 'https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_multiclass_256x256/float32/latest/' + gfile
        urllib.request.urlretrieve(src, gfile) if src.startswith('http') else __import__('shutil').copy(src, gfile)
    it = tf.lite.Interpreter(model_path=gfile); it.allocate_tensors()
    gi, go = it.get_input_details()[0], it.get_output_details()[0]
    REMAP = np.array([0, 2, 0, 1, 0, 0], np.uint8)          # 구글: 0배경 1머리 2몸피부 3얼굴피부 4옷 5기타 → 우리 번호
    G = np.zeros_like(Yva)
    for i in range(len(Xva)):
        it.set_tensor(gi['index'], Xva_in[i:i + 1]); it.invoke()
        G[i] = REMAP[it.get_tensor(go['index'])[0].argmax(-1)]
    for c, name in enumerate(CLASS_NAMES[:3]):
        inter = ((G == c) & (Yva == c)).sum(); union = ((G == c) | (Yva == c)).sum()
        print('%-5s  ours %.3f   google %.3f' % (name, ious[name], inter / max(union, 1)))

## 6. TFLite 변환 (양자화 없이) · 검증 · 저장

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
# conv.optimizations = [tf.lite.Optimize.DEFAULT]     ← 켜지 않음 (보드 런타임 호환)
tfl = conv.convert()
open(save_dir + 'hair_seg.tflite', 'wb').write(tfl)
print('hair_seg.tflite  %.2f MB' % (len(tfl) / 1e6))
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
i_, o_ = it.get_input_details()[0], it.get_output_details()[0]
agree = []
for k in range(20):
    it.set_tensor(i_['index'], Xva_in[k:k + 1]); it.invoke()
    agree.append((it.get_tensor(o_['index'])[0].argmax(-1) == P[k]).mean())
print('keras vs tflite 픽셀 일치율: %.4f' % np.mean(agree))
for f in ['hair_seg.tflite', 'hairseg_train_log.png', 'hairseg_samples.png']:
    print('%-24s %s' % (f, 'OK' if os.path.exists(save_dir + f) else '없음'))

## 7. 보드 적용
`save/hair_seg.tflite` 를 보드 폴더에 넣고, `face_fortune.py` 의 `[3] hairline` 구역에서 `Segmenter` 를 LiteRT 로 직접 여는 버전으로 바꿉니다
(상수 `FACE_SKIN=1, HAIR=2`, 얼굴 주변 크롭). 코드는 안내 HTML 7절.

## 8. (선택) WFLW 점 모델 교차검증 — 학습 없음
CelebAMask-HQ 검증 사진에 우리 `wflw_mesh.tflite` 를 돌려, 눈 점이 눈 마스크 안에, 입 점이 입술 마스크 안에 들어가는 비율을 셉니다.
`proj_face/FaceReading_3/` 의 `wflw_prep.py`, `wflw_mesh.py` 와 `save/` 의 `wflw_mesh.tflite`, `face_detection_yunet_2023mar.onnx` 가 필요합니다.

In [ ]:
RUN_XVAL = True
if RUN_XVAL:
    import shutil
    for f in ['wflw_prep.py', 'wflw_mesh.py']: shutil.copy(CODE_DIR + f, './')
    for f in ['wflw_mesh.tflite', 'face_detection_yunet_2023mar.onnx']: shutil.copy(save_dir + f, './')
    sys.path.insert(0, '.')
    from wflw_mesh import WflwMesh
    mesh = WflwMesh()
    def part_mask(idx, names):
        m = np.zeros((512, 512), bool); folder = ANNO + '%d/' % (idx // 2000)
        for n in names:
            p = folder + '%05d_%s.png' % (idx, n)
            if os.path.exists(p): m |= cv2.imread(p, 0) > 0
        return m
    EYE_L, EYE_R, LIPS = list(range(60, 68)) + [96], list(range(68, 76)) + [97], list(range(76, 96))
    hits = {'eye_l': [], 'eye_r': [], 'lips': []}; noface = 0; K = np.ones((7, 7), np.uint8)
    for idx in range(N - NVAL, N):
        img = cv2.resize(cv2.imread(IMG_DIR + '%d.jpg' % idx), (512, 512))
        pts = mesh(img)
        if pts is None: noface += 1; continue
        for key, ids, names in (('eye_l', EYE_L, ['l_eye']), ('eye_r', EYE_R, ['r_eye']), ('lips', LIPS, ['u_lip', 'l_lip', 'mouth'])):
            m = cv2.dilate(part_mask(idx, names).astype(np.uint8), K) > 0          # 경계 3px 여유
            xy = np.clip(pts[ids, :2].astype(int), 0, 511)
            hits[key].append(m[xy[:, 1], xy[:, 0]].mean())
    print('얼굴 못 찾음:', noface, '/', NVAL)
    for k, v in hits.items(): print('%-6s 점이 마스크 안에 들어간 비율: %.1f%%' % (k, 100 * np.mean(v)))
    print('(l_eye 와 eye_l 수치가 서로 바뀌어 보이면 데이터셋의 좌우 명명 차이 — 이름만 바꿔 읽으면 됨)')